# Transfer Learning

Treinar uma CNN do zero exige muitas imagens. Com poucas centenas de exemplos, a rede decora o treino e erra em imagens novas.

No **transfer learning**, começamos com uma rede que já foi treinada em um dataset enorme, o **ImageNet** (1,2 milhão de fotos, 1000 classes), e reaproveitamos o que ela aprendeu. As camadas iniciais dessa rede detectam bordas, cores e texturas, e as seguintes combinam isso em partes de objetos. Isso serve para quase qualquer problema com fotos. Só a última camada, que escolhe entre as 1000 classes do ImageNet, é específica da tarefa original.

Vamos resolver o mesmo problema de duas formas e comparar: treinando uma CNN do zero e reaproveitando uma rede pré-treinada.

In [ ]:
import os
import zipfile
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Dados

O dataset **Hymenoptera** tem fotos de formigas e abelhas: cerca de 120 de cada classe para treino e 75 para validação. É pouco para treinar uma CNN do zero.

In [ ]:
if not os.path.exists("data/hymenoptera_data"):
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    response = requests.get(url)
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        z.extractall("data")

As imagens estão em uma pasta por classe:

```
data/hymenoptera_data/
    train/ants/  train/bees/
    val/ants/    val/bees/
```

Para esse formato, o torchvision já tem um `Dataset` pronto, o `datasets.ImageFolder`: cada subpasta vira uma classe.

As transformações repetem as que a rede pré-treinada viu no ImageNet: recorte de $224 \times 224$ e normalização com a média e o desvio padrão de cada canal do ImageNet. No treino, o recorte e o espelhamento são aleatórios (aumento de dados). Na validação, o recorte é sempre o central.

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_set = datasets.ImageFolder("data/hymenoptera_data/train", transform=train_transform)
val_set = datasets.ImageFolder("data/hymenoptera_data/val", transform=val_transform)

train_dataloader = DataLoader(train_set, batch_size=16, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_set, batch_size=64, num_workers=2)

classes = train_set.classes
print(classes)
print(len(train_set), len(val_set))

Para mostrar as imagens, desfazemos a normalização.

In [ ]:
def denormalize(image):
    image = image.permute(1, 2, 0)               # [3, H, W] -> [H, W, 3]
    image = image * torch.tensor(std) + torch.tensor(mean)
    return image.clamp(0, 1)


images, labels = next(iter(train_dataloader))
print(images.shape)   # [16, 3, 224, 224]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(denormalize(image))
    ax.set_title(classes[label])
    ax.axis("off")
plt.show()

## Funções de treino

As mesmas dos materiais anteriores, com duas mudanças:

- a acurácia é medida no conjunto de validação;
- o otimizador recebe só os parâmetros que vão ser treinados. Isso vai importar quando congelarmos parte da rede.

In [ ]:
criterion = nn.CrossEntropyLoss()


def accuracy(model, dataloader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    return correct / len(dataloader.dataset)


def train(model, train_dataloader, val_dataloader, epochs, lr=1e-3):
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_dataloader)
        val_accuracy = accuracy(model, val_dataloader)
        print(f"[{epoch + 1}/{epochs}] perda {mean_loss:.4f}, acurácia de validação {val_accuracy:.3f}")

    return val_accuracy


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## CNN do zero

Primeiro, uma referência: uma CNN pequena, como as do material anterior, treinada do zero.

No fim da parte convolucional, o `nn.AdaptiveAvgPool2d(1)` tira a média de cada mapa de características e deixa um único valor por canal. A ResNet usa o mesmo recurso antes da camada linear.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),    # [N, 32, 224, 224]
            nn.ReLU(),
            nn.MaxPool2d(4),                               # [N, 32, 56, 56]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # [N, 64, 56, 56]
            nn.ReLU(),
            nn.MaxPool2d(4),                               # [N, 64, 14, 14]
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # [N, 128, 14, 14]
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                       # [N, 128, 1, 1]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                  # [N, 128]
            nn.Linear(128, 2),                             # [N, 2]
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
torch.manual_seed(42)
small_cnn = SmallCNN().to(device)
small_cnn_accuracy = train(small_cnn, train_dataloader, val_dataloader, epochs=15)

A rede fica entre 60% e 70% de acurácia, pouco acima do acaso (50%). Com 244 imagens de treino, não há exemplos suficientes para aprender do zero o que diferencia uma formiga de uma abelha.

## ResNet pré-treinada

O `torchvision.models` traz arquiteturas conhecidas, como VGG, ResNet e EfficientNet, com pesos já treinados no ImageNet. Vamos usar a **ResNet-18**. O objeto `weights` guarda os pesos e também os nomes das 1000 classes.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

print(f"{count_parameters(resnet):,} parâmetros")
print(resnet.fc)

A última camada, `fc`, recebe 512 características e devolve 1000 logits, um por classe do ImageNet. Todo o resto da rede serve para extrair essas 512 características.

Sem nenhum treino, a rede já reconhece as classes do ImageNet. A imagem passa pelas mesmas transformações da validação.

In [ ]:
url = "https://www.uni-jena.de/unijenamedia/387585/elefant.jpg?height=428&width=760"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
x = val_transform(image).unsqueeze(0)   # [1, 3, 224, 224]

resnet.eval()
with torch.no_grad():
    logits = resnet(x)                  # [1, 1000]

probabilities = logits[0].softmax(dim=0)
top_probabilities, top_classes = probabilities.topk(5)

for p, c in zip(top_probabilities, top_classes):
    print(f"{weights.meta['categories'][c]:25s} {p:.3f}")

## Rede congelada

"Formiga" e "abelha" não estão entre as classes que a ResNet devolve, mas as características que ela extrai servem para separá-las. Então fazemos duas coisas:

1. **congelamos** a rede inteira, para que os pesos pré-treinados não mudem;
2. trocamos a `fc` por uma camada nova, com 2 saídas, e treinamos só ela.

`requires_grad_(False)` congela todos os parâmetros de um módulo: o backpropagation não calcula gradientes para eles. A `nn.Linear` nova é criada depois, então ela continua treinável.

In [ ]:
frozen = models.resnet18(weights=weights)
frozen.requires_grad_(False)

frozen.fc = nn.Linear(frozen.fc.in_features, len(classes))   # 512 -> 2
frozen = frozen.to(device)

print(f"{count_parameters(frozen):,} parâmetros, {count_trainable(frozen):,} treináveis")

Só 1026 parâmetros são treinados: $512 \times 2$ pesos e 2 vieses. Na prática, é uma regressão logística sobre as características da ResNet.

In [ ]:
torch.manual_seed(42)
frozen_accuracy = train(frozen, train_dataloader, val_dataloader, epochs=5)

## Comparação

In [ ]:
print(f"CNN do zero:         {count_trainable(small_cnn):6,} parâmetros treináveis, acurácia {small_cnn_accuracy:.3f}")
print(f"ResNet-18 congelada: {count_trainable(frozen):6,} parâmetros treináveis, acurácia {frozen_accuracy:.3f}")

Treinando 90 vezes menos parâmetros, e por um terço das épocas, a rede pré-treinada passa de 90% de acurácia. Algumas previsões dela (em vermelho, os erros):

In [ ]:
images, labels = next(iter(val_dataloader))

frozen.eval()
with torch.no_grad():
    predictions = frozen(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, i in zip(axes.flat, torch.randperm(len(images))):
    ax.imshow(denormalize(images[i]))
    color = "green" if predictions[i] == labels[i] else "red"
    ax.set_title(f"previsto: {classes[predictions[i]]}, real: {classes[labels[i]]}", color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Salvar e carregar

Para usar o modelo depois sem treinar de novo, salvamos o `state_dict`: um dicionário com o tensor de cada parâmetro.

In [ ]:
torch.save(frozen.state_dict(), "ants_bees.pth")

O arquivo guarda só os pesos, não a arquitetura. Para carregar, montamos a mesma arquitetura (ResNet-18 com a `fc` de 2 saídas) e copiamos os pesos para ela. Não precisamos dos pesos do ImageNet: todos já estão no arquivo.

In [ ]:
loaded = models.resnet18()
loaded.fc = nn.Linear(loaded.fc.in_features, len(classes))
loaded.load_state_dict(torch.load("ants_bees.pth", weights_only=True))
loaded = loaded.to(device)

print(f"acurácia {accuracy(loaded, val_dataloader):.3f}")   # a mesma do modelo salvo

## Exercícios

### Exercício 1

No **ajuste fino** (*fine-tuning*), além da camada nova, treinamos também as últimas camadas da rede pré-treinada. A ResNet-18 tem quatro estágios, `layer1` a `layer4`, seguidos da `fc`. Congele a rede inteira, descongele o `layer4` com `requires_grad_(True)`, troque a `fc` e treine por 5 épocas com `lr=1e-4`. A acurácia melhora em relação à rede congelada? Depois, repita com `lr=1e-3`: o que acontece nas primeiras épocas, e por quê?

In [ ]:
finetuned = models.resnet18(weights=weights)

### Exercício 2

Escolha outra arquitetura na [lista do torchvision](https://docs.pytorch.org/vision/main/models.html#classification), como a `mobilenet_v3_small` ou a `efficientnet_b0`, e repita o treino com a rede congelada. A última camada não se chama `fc` em todas as arquiteturas: use `print` no modelo para encontrá-la. Compare acurácia e número de parâmetros com a ResNet-18.

In [ ]:
# other = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

### Exercício 3

Treine a ResNet-18 congelada com só 40 imagens de treino, sorteadas do `train_set`. Ela ainda supera a CNN treinada do zero com as 244 imagens?

In [ ]:
indices = torch.randperm(len(train_set))[:40]
small_train_set = torch.utils.data.Subset(train_set, indices)